In [ ]:
import numpy as np
import pandas as pd
import requests
import duckdb
import os
import s3fs
import sys
sys.path.append('../outils')

import fonctions_nomenclature as fnom
import fonctions_preparation_table as fprep

np.random.seed(123)

# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})
BUCKET = "mathildedonnenfeld/diffusion/bistrots_et_bulletins"

In [ ]:
FILE_KEY_S3 = "t1_muni.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    t1_muni = pd.read_csv(file_in, sep=",")

t1_muni.head()

In [ ]:
FILE_KEY_S3 = "t2_muni.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    t2_muni = pd.read_csv(file_in, sep=",")

t2_muni.head()

In [ ]:
FILE_KEY_S3 = "bistrots_sirene.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    bistrots_sirene = pd.read_csv(file_in, sep=",")

bistrots_sirene.head()

In [ ]:
FILE_KEY_S3 = "filosofi.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    filosofi = pd.read_csv(file_in, sep=",")

filosofi.head()

## Explo et prep données bistrots 

In [ ]:
bistrots_sirene.describe()

In [ ]:
# Data prep Sirene ----------------------------
## Modifier le format de la dateDernierTraitementEtablissement (enlever l'heure) et de la dateCreationEtablissement 
bistrots_sirene["date_dernier_traitement"] = pd.to_datetime(bistrots_sirene.dateDernierTraitementEtablissement).dt.normalize()
bistrots_sirene["date_creation"] = pd.to_datetime(bistrots_sirene.dateCreationEtablissement).dt.normalize()
bistrots_sirene.head()

In [ ]:
bistrots_sirene["date_dernier_traitement"].describe()

In [ ]:
bistrots_sirene["date_creation"].describe()

In [ ]:
## On étudie la variable dateCreationEtablissement  => quelle est la dernière date de création 
# max : 2027-02-01 => étrange, pourquoi i) des dates dans le futur, et ii) la date de mise à jour la plus récente est moins récente (cf cellule plus bas) ??
# min : 1900-01-01 (évidement oui, également la date d'entrée en vigueur de code civil allemand fyi)


In [ ]:
##  Garder uniquement les établissements actifs à une date t
### On peut choisir arbitrairement le 1er janvier de l'année des élections municipales comme condition
cond_date = "2026-01-01"
print(cond_date)
fraise = bistrots_sirene.copy()

fraise = fraise[(fraise['dateCreationEtablissement'] <= cond_date)]

# compter le nb qu'il reste => la différence est huge, vérifier l'output voir si les conditions sont les bonnes (ou trop restrictives)
print(bistrots_sirene["siret"].nunique())
print(fraise["siret"].nunique())

In [ ]:
# Proposition code alternatif pour décomptes sans copie
cond_date = "2026-01-01"
print(bistrots_sirene["siret"].nunique())
print(bistrots_sirene[(bistrots_sirene['dateCreationEtablissement'] <= cond_date)]["siret"].nunique())

In [ ]:
bistrots_par_ville = (
    bistrots_sirene
    .groupby(["codeCommuneEtablissement", "libelleCommuneEtablissement"])["siret"]
    .nunique()
    .rename("nb_etablissements")
    .reset_index()
    .sort_values("nb_etablissements", ascending=False)
)

In [ ]:
bistrots_par_ville.head(20)

## Explo et prep données municipales

On cherche à construire 3 indicateurs : 

1°) Le vote simili Attal: 1 si la nuance majoritaire au conseil municipal est de droite, 0 sinon (booléen)

2°) La majorité simili Attal : 1 si la couleur majoritaire au conseil municipal est de droite, 0 sinon (indépendamment de la liste arrivée en tête)((booléen)

3°) Le vote d'extrême droite : 1 si la nuance majoritaire au conseil municipal est d'extrême droite, 0 sinon (booléen)

Nous allons d'abord préparer la table du deuxième tour, qui contient uniquement les villes pour lesquelles il y a eu deux tours d'élection.
Ainsi, nous pourrons récupérer les codes communes de ces villes, pour les enlever de la table du premier tour et ne traiter que les villes n'ayant eu qu'un tour. 

PS : dans un deuxième temps, peut être qu'on reviendra sur ce traitement pour ajouter des indicateurs sur le nombre de tours, le nombre de listes etc. 

### Préparation de la table du deuxième tour
On commence par traiter les villes ayant eu deux tours aux élections municipales

In [ ]:
t2_muni.head(20)

In [ ]:
t2_muni.describe()

In [ ]:
t2_muni.columns #=> trop de colonnes, il faut sélectionner les colonnes pertinentes 

In [ ]:
# Synthèse de la table du deuxième tour ------------------------
col_com = t2_muni.columns[~ t2_muni.columns.str.contains(r"\d")].tolist()

## On garde certaines informations sur l'issue du vote : nuance et nombre de siège au conseil municipal (CM)
col_vote = t2_muni.columns[t2_muni.columns.str.contains(r"Nuance|CM", case=False)].tolist()

## Concaténer et sélectionner
t2_muni=t2_muni[col_com + col_vote]

## On pivote l'information de vote en se basant sur 2 colonnes uniquement 
t2_muni_long = (pd.wide_to_long(
        t2_muni,
        stubnames=["Nuance liste", "Sièges au CM"],
        i="Code commune",
        j="numero_liste",
        sep=" ",
        suffix=r"\d+")
    .rename(columns={"Nuance liste": "Nuance"})
    .dropna(subset=["Nuance"]) ## ATTENTION CETTE LIGNE !!!!! 
    .reset_index()
    .drop(columns="numero_liste"))

t2_muni_long.head(10)


In [ ]:
# Création d'une nomenclature associant une couleur à une nuance ------------

## Récupérer les nuances
label_nuance = pd.DataFrame({"Nuance": t2_muni_long["Nuance"].unique()})
# print(label_nuance)

nomenclature_couleur_nuance = fnom.classer_nuance(label_nuance) #fonction maison
print(nomenclature_couleur_nuance)

In [ ]:
# Création indicateur 1 ----------------------------------

# Ajout du nombre de listes/candidats par ville au second tour
t2_muni_long['nombre_candidats_t2'] = (t2_muni_long.groupby('Code commune')['Nuance'].transform('size'))
t2_muni_long.head()

## test pour une commune 
#visu = t2_muni_long[t2_muni_long['Code commune'] == '01249']
#visu.head()

## Nuance de la liste élue + raccourcir table
t2_muni_ind1 = (t2_muni_long.loc[t2_muni_long.groupby('Code commune')['Sièges au CM'].idxmax(),
        ['Code commune', 'Libellé commune', 'nombre_candidats_t2', 'Nuance']]
    .drop_duplicates()
).reset_index(drop=True)

## Ajout nomenclature
t2_muni_ind1 = t2_muni_ind1.merge(nomenclature_couleur_nuance).rename(columns={'Nuance': 'Nuance_maj'})

## Indicateur 1 
t2_muni_ind1['ind1']  = t2_muni_ind1['ind1'] = (t2_muni_ind1['Couleur'] == 'Droite').astype(int)

t2_muni_ind1['Nuance_maj'].value_counts()
t2_muni_ind1['ind1'].value_counts()

t2_muni_ind1.head()


In [ ]:
# Création indicateur 2 ----------------------------------

## Ajout nomenclature 
t2_muni_long= t2_muni_long.merge(nomenclature_couleur_nuance)

# Sommer le nombre de sièges au conseil municipal par commune et couleur, et garder la couleur majoritaire

t2_muni_ind2 = (
    t2_muni_long.drop(columns='Nuance')
    .groupby(['Code commune', 'Couleur'], as_index=False)
    .agg({
        'Sièges au CM': 'sum'})
)

t2_muni_ind2 = t2_muni_ind2.loc[t2_muni_ind2.groupby('Code commune')['Sièges au CM'].idxmax()]

t2_muni_ind2["ind2"] = (t2_muni_ind2["Couleur"] == "Droite").astype(int)

In [ ]:
# Création indicateur 3 ----------------------------------

t2_muni_ind3 = fnom.classer_nuance_extd(t2_muni_ind1.drop(columns=["ind1", "Couleur"]))

t2_muni_ind3['ind3'].value_counts()

In [ ]:
# Table synthétique contenant les 3 indicateurs --------

t2_muni_ind = (
    t2_muni_ind1
    .drop(columns=["Nuance_maj", "Couleur"])
    .merge(t2_muni_ind2[["Code commune", "ind2"]], on="Code commune", how="left")
    .merge(t2_muni_ind3[["Code commune", "ind3"]], on="Code commune", how="left")
)

t2_muni_ind.head()

In [ ]:
# Récupérer les codes communes des communes ayant eu deux tours d'élection 

cc_2t = t2_muni_ind["Code commune"].unique()
#cc_2t = t2_muni_ind["Code commune"].unique() #937
print(len(cc_2t)) #les deux lignes ci-dessus donnent le même nombre de code commune

to do :
vérifier qu'ils sont cohérents : si ind3 = 1, ind1 = 1 aussi. Voir la freq de ind1 et ind2, ils sont très fortement corrélés normalement. 

### Préparation de la table du premier tour

In [ ]:
### TESTS : A SUPPRIMER A LA FIN
## y a qui dans cette table : est-ce que les villes qui ont eu 2 tours y sont ? 
###Ex : 1283 (voir t2)
yaki = t1_muni[t1_muni["Code commune"] == 1283]
print(yaki)

#Oui, ils y sont => voir quelle méthode on peut faire : récup les communes en T2 et les enlever de T1 ? 
#rend possible l'utilisation du code de T2 pour nettoyer T1. 


In [ ]:
# Selection des villes n'ayant eu qu'un seul tour d'élection (en pratique : on enlève celles qui en ont eu 2)

## Mise au format des codes commune dans la table t1 : rajouter un 0 devant les codes commune avec 4 chiffres
t1_muni["Code commune"]= t1_muni["Code commune"].astype(str).str.zfill(5)
## Sélection inversée 
t1_muni_keep = t1_muni[~t1_muni["Code commune"].isin(cc_2t)] 

# Vérification : a-t-on toutes les villes ? Oui ! Bravoo, la bataille contre le formatage des codes communes est gagnée
print(t1_muni["Code commune"].unique().shape[0]) # 34836
print(t1_muni_keep["Code commune"].unique().shape[0] +len(cc_2t)) #34836

In [ ]:
## Les tables ont-elles la même structure ?
#print("Colonnes dans t1_muni uniquement :", t1_muni.columns.difference(t2_muni.columns).tolist())
#print("Colonnes dans t2_muni uniquement :", t2_muni.columns.difference(t1_muni.columns).tolist())

colonnes_communes = t1_muni.columns.intersection(t2_muni.columns).tolist()
print(colonnes_communes) #il me semble qu'on peut appliquer le code de t1. 

In [ ]:
# Préparation de T1 à l'image de T2 : zé parti 

t1_muni_long = fprep.synthese_table_muni(t1_muni_keep)
print(len(t1_muni["Code commune"].unique()))
print(len(t1_muni_long["Code commune"].unique())) # pas égal, à creuser. 

### Jointure des tables des deux tours

In [ ]:
#todo

# TODO data prep

- Filtrer les bistrots actifs à une date t que l'on déterminera [en cours, besoin de se renseigner sur la façon dont est remplie la variable dateDernierTraitementEtablissement] OK
- Agréger les bistrots Sirene en un compte de bistrots par commune OK
- Traiter les données des municipales pour obtenir un indicateur quanti du vote "simili-Attal" (majo présidentielle, en gros) et du vote RN [EN COURS]
    - Table T1 : [En cours]
        - Garder le nb de communes qui ont eu 1 seul tour uniquement (fait)
        - Traiter ces communes en factorisant le code de T2 dans des fonctions 
    - Table T2 : en cours + à vérifier : 
        - Regarder ce que fais la ligne .dropna(subset=["Nuance"]) => supprime les codes commune sans nuance renseignée. Cb de communes cela concerne ? comment gérer le pb ? garder que la colonne qui finit en 1 et mettre "autre nuance" ? Que signifie une absence de nuance ? une liste indépendante ? 
        - Après création de T1 grace à des fonctions ad hoc adaptées du traitement de T2, utiliser ces fonctions sur T2
    - Fusionner les tables T1 et T2 propres
- Sélectionner les variables d'intérêt dans chaque table pour en faire quelque chose de lisible
- Fusionner les tables (partir des tables T1 et T2 fusionnées comme référence)